In [2]:
# ============================================================
# MATERIALS PROJECT DATA CLEANING
# File: materials_full_structure_data.csv
# ============================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime
from pymatgen.core import Composition

# ============================================================
# STEP 1: FILE SETTINGS
# ============================================================

INPUT_FILE = "materials_full_structure_data.csv"
OUTPUT_DIR = "materials_project_cleaning"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("MATERIALS PROJECT DATA CLEANING")
print("=" * 70)


# ============================================================
# STEP 2: READ CSV
# ============================================================

print("\nReading file...")

df = pd.read_csv(INPUT_FILE)

print("File loaded successfully!")

print("\nOriginal columns:")
for i, col in enumerate(df.columns):
    print(f"{i}: {repr(col)}")

print("\nInitial shape:")
print(df.shape)


# ============================================================
# STEP 3: STANDARDIZE COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nStandardized columns:")

for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


# ============================================================
# STEP 4: AUTOMATIC COLUMN DETECTION
# ============================================================

def find_column(columns, exact_names, keywords):

    # First try exact names
    for name in exact_names:
        if name in columns:
            return name

    # Then search using keywords
    for col in columns:

        for keyword in keywords:

            if keyword in col:
                return col

    return None


# ------------------------------------------------------------
# MATERIAL ID
# ------------------------------------------------------------

material_id_col = find_column(
    df.columns,

    [
        "material_id",
        "materialid",
        "mp_id",
        "mpid",
        "material"
    ],

    [
        "material_id",
        "materialid",
        "mp_id",
        "mpid"
    ]
)


# ------------------------------------------------------------
# FORMULA
# ------------------------------------------------------------

formula_col = find_column(
    df.columns,

    [
        "formula",
        "formula_pretty",
        "pretty_formula",
        "chemical_formula",
        "composition",
        "formula_reduced",
        "reduced_formula"
    ],

    [
        "formula",
        "composition"
    ]
)


# ------------------------------------------------------------
# STRUCTURE
# ------------------------------------------------------------

structure_col = find_column(
    df.columns,

    [
        "structure",
        "structure_json",
        "structure_data",
        "crystal_structure"
    ],

    [
        "structure"
    ]
)


# ============================================================
# STEP 5: DISPLAY DETECTED COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("AUTOMATIC COLUMN DETECTION")
print("=" * 70)

print("Material ID column :", material_id_col)
print("Formula column     :", formula_col)
print("Structure column   :", structure_col)


# ============================================================
# STEP 6: IF FORMULA WAS NOT FOUND
# ============================================================

if formula_col is None:

    print("\nWARNING: Formula column was not detected.")

    print("\nYour available columns are:")

    for col in df.columns:
        print(" ->", col)

    # --------------------------------------------------------
    # Try to identify formula from actual data
    # --------------------------------------------------------

    print("\nSearching columns for chemical-formula-like data...")

    formula_candidates = []

    for col in df.columns:

        # Ignore obvious non-formula columns
        if col == material_id_col:
            continue

        if col == structure_col:
            continue

        sample = (
            df[col]
            .dropna()
            .astype(str)
            .head(100)
        )

        if len(sample) == 0:
            continue

        valid_count = 0

        for value in sample:

            try:

                comp = Composition(value)

                if len(comp.elements) > 0:
                    valid_count += 1

            except Exception:
                pass

        if valid_count > 0:

            percentage = (
                valid_count / len(sample) * 100
            )

            formula_candidates.append(
                (col, percentage)
            )

    if formula_candidates:

        formula_candidates.sort(
            key=lambda x: x[1],
            reverse=True
        )

        print("\nPossible formula columns:")

        for col, percentage in formula_candidates:
            print(
                f" -> {col}: "
                f"{percentage:.1f}% formula-like values"
            )

        # Automatically select the best candidate
        formula_col = formula_candidates[0][0]

        print(
            "\nAutomatically selected formula column:",
            formula_col
        )

    else:

        raise ValueError(
            "\nCould not automatically identify a "
            "chemical formula column.\n\n"
            "Please check the printed column names above."
        )


# ============================================================
# STEP 7: MATERIAL ID CHECK
# ============================================================

if material_id_col is None:

    print(
        "\nWARNING: Material ID column was not detected."
    )

    print("Available columns:")

    for col in df.columns:
        print(" ->", col)

    raise ValueError(
        "Material ID column could not be identified."
    )


# ============================================================
# STEP 8: BACKUP RAW DATA
# ============================================================

raw_backup = os.path.join(
    OUTPUT_DIR,
    "raw_backup.csv"
)

df.to_csv(
    raw_backup,
    index=False
)

print("\nRaw backup saved:")
print(raw_backup)


# ============================================================
# STEP 9: INITIAL STATISTICS
# ============================================================

initial_rows = len(df)
initial_columns = len(df.columns)

report = []

report.append(
    "MATERIALS PROJECT DATA CLEANING REPORT"
)

report.append("=" * 70)

report.append(
    f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

report.append("")

report.append("INITIAL DATASET")
report.append("-" * 70)

report.append(
    f"Initial records: {initial_rows}"
)

report.append(
    f"Initial columns: {initial_columns}"
)

report.append(
    f"Material ID column: {material_id_col}"
)

report.append(
    f"Formula column: {formula_col}"
)

report.append(
    f"Structure column: {structure_col}"
)


# ============================================================
# STEP 10: STANDARDIZE MISSING VALUES
# ============================================================

missing_tokens = [
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none",
    "NaN",
    "nan"
]

df = df.replace(
    missing_tokens,
    np.nan
)


# ============================================================
# STEP 11: MISSING VALUES BEFORE CLEANING
# ============================================================

missing_before = df.isna().sum()

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE CLEANING")
print("=" * 70)

print(
    missing_before[
        missing_before > 0
    ]
)

report.append("")
report.append(
    "MISSING VALUES BEFORE CLEANING"
)

report.append("-" * 70)

for col, count in missing_before.items():

    if count > 0:

        percentage = (
            count / initial_rows * 100
        )

        report.append(
            f"{col}: {count} "
            f"({percentage:.2f}%)"
        )


# ============================================================
# STEP 12: EXACT DUPLICATES
# ============================================================

duplicate_rows = df.duplicated().sum()

print(
    "\nExact duplicate rows:",
    duplicate_rows
)

report.append("")
report.append("EXACT DUPLICATES")
report.append("-" * 70)

report.append(
    f"Duplicate rows found: {duplicate_rows}"
)


# ============================================================
# STEP 13: REMOVE EXACT DUPLICATES
# ============================================================

before = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

removed_exact_duplicates = (
    before - len(df)
)

report.append(
    f"Exact duplicate rows removed: "
    f"{removed_exact_duplicates}"
)


# ============================================================
# STEP 14: DUPLICATE MATERIAL IDs
# ============================================================

duplicate_ids = (
    df[material_id_col]
    .duplicated(keep=False)
    .sum()
)

print(
    "Records with duplicate material IDs:",
    duplicate_ids
)

report.append("")
report.append("DUPLICATE MATERIAL IDs")
report.append("-" * 70)

report.append(
    f"Duplicate material ID records: "
    f"{duplicate_ids}"
)


# ============================================================
# STEP 15: SAVE DUPLICATE IDs FOR AUDIT
# ============================================================

duplicate_id_data = df[
    df[material_id_col]
    .duplicated(keep=False)
]

if len(duplicate_id_data) > 0:

    duplicate_id_file = os.path.join(
        OUTPUT_DIR,
        "duplicate_material_ids.csv"
    )

    duplicate_id_data.to_csv(
        duplicate_id_file,
        index=False
    )


# ============================================================
# STEP 16: REMOVE DUPLICATE MATERIAL IDs
# ============================================================

before = len(df)

df = df.drop_duplicates(
    subset=[material_id_col],
    keep="first"
)

df = df.reset_index(drop=True)

removed_duplicate_ids = (
    before - len(df)
)

report.append(
    f"Duplicate material ID records removed: "
    f"{removed_duplicate_ids}"
)


# ============================================================
# STEP 17: REMOVE MISSING MATERIAL IDs
# ============================================================

before = len(df)

df = df.dropna(
    subset=[material_id_col]
)

df = df.reset_index(drop=True)

removed_missing_ids = (
    before - len(df)
)

report.append(
    f"Missing material ID records removed: "
    f"{removed_missing_ids}"
)


# ============================================================
# STEP 18: REMOVE MISSING FORMULAS
# ============================================================

before = len(df)

df = df.dropna(
    subset=[formula_col]
)

df = df.reset_index(drop=True)

removed_missing_formulas = (
    before - len(df)
)

report.append(
    f"Missing formula records removed: "
    f"{removed_missing_formulas}"
)


# ============================================================
# STEP 19: VALIDATE CHEMICAL FORMULAS
# ============================================================

print("\nValidating chemical formulas...")


def validate_formula(value):

    try:

        if pd.isna(value):
            return False

        value = str(value).strip()

        if value == "":
            return False

        composition = Composition(value)

        if len(composition.elements) == 0:
            return False

        return True

    except Exception:

        return False


df["formula_valid"] = (
    df[formula_col]
    .apply(validate_formula)
)

invalid_formula_count = (
    (~df["formula_valid"]).sum()
)

print(
    "Invalid formulas:",
    invalid_formula_count
)

report.append("")
report.append("CHEMICAL FORMULA VALIDATION")
report.append("-" * 70)

report.append(
    f"Invalid formulas: "
    f"{invalid_formula_count}"
)


# ============================================================
# STEP 20: SAVE INVALID FORMULAS
# ============================================================

invalid_formulas = df[
    ~df["formula_valid"]
]

if len(invalid_formulas) > 0:

    invalid_formula_file = os.path.join(
        OUTPUT_DIR,
        "invalid_formulas.csv"
    )

    invalid_formulas.to_csv(
        invalid_formula_file,
        index=False
    )


# ============================================================
# STEP 21: REMOVE INVALID FORMULAS
# ============================================================

before = len(df)

df = df[
    df["formula_valid"]
].copy()

df = df.drop(
    columns=["formula_valid"]
)

df = df.reset_index(drop=True)

removed_invalid_formulas = (
    before - len(df)
)

report.append(
    f"Invalid formula records removed: "
    f"{removed_invalid_formulas}"
)


# ============================================================
# STEP 22: STRUCTURE CHECK
# ============================================================

report.append("")
report.append("STRUCTURE DATA")
report.append("-" * 70)

if structure_col is not None:

    missing_structures = (
        df[structure_col].isna().sum()
    )

    print(
        "\nMissing structures:",
        missing_structures
    )

    report.append(
        f"Missing structures: "
        f"{missing_structures}"
    )

    # Remove empty structure values
    before = len(df)

    df = df.dropna(
        subset=[structure_col]
    )

    df = df[
        df[structure_col]
        .astype(str)
        .str.strip()
        .notna()
    ]

    df = df[
        ~df[structure_col]
        .astype(str)
        .str.strip()
        .isin([
            "",
            "nan",
            "None",
            "null"
        ])
    ]

    df = df.reset_index(drop=True)

    removed_structures = (
        before - len(df)
    )

    report.append(
        f"Records removed due to missing/empty "
        f"structures: {removed_structures}"
    )

else:

    report.append(
        "Structure column was not found."
    )


# ============================================================
# STEP 23: FINAL CHECK
# ============================================================

final_duplicate_rows = (
    df.duplicated().sum()
)

final_duplicate_ids = (
    df[material_id_col]
    .duplicated()
    .sum()
)

final_missing = df.isna().sum()

final_rows = len(df)

total_removed = (
    initial_rows - final_rows
)

percentage_removed = (
    total_removed / initial_rows * 100
    if initial_rows > 0
    else 0
)


# ============================================================
# STEP 24: FINAL REPORT
# ============================================================

report.append("")
report.append("FINAL QUALITY CHECK")
report.append("-" * 70)

report.append(
    f"Final records: {final_rows}"
)

report.append(
    f"Records removed: {total_removed}"
)

report.append(
    f"Percentage removed: "
    f"{percentage_removed:.2f}%"
)

report.append(
    f"Remaining duplicate rows: "
    f"{final_duplicate_rows}"
)

report.append(
    f"Remaining duplicate material IDs: "
    f"{final_duplicate_ids}"
)

report.append(
    f"Remaining missing values: "
    f"{final_missing.sum()}"
)


if (
    final_duplicate_rows == 0
    and final_duplicate_ids == 0
    and final_missing.sum() == 0
):

    status = "PASSED"

else:

    status = "REQUIRES REVIEW"


report.append("")
report.append("DATA QUALITY STATUS")
report.append("-" * 70)

report.append(
    f"Status: {status}"
)


# ============================================================
# STEP 25: SAVE CLEANED DATA
# ============================================================

cleaned_file = os.path.join(
    OUTPUT_DIR,
    "materials_cleaned.csv"
)

df.to_csv(
    cleaned_file,
    index=False
)


# ============================================================
# STEP 26: SAVE REPORT
# ============================================================

report_file = os.path.join(
    OUTPUT_DIR,
    "formal_data_cleaning_report.txt"
)

with open(
    report_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report)
    )


# ============================================================
# STEP 27: SAVE REPORT AS CSV
# ============================================================

report_csv = os.path.join(
    OUTPUT_DIR,
    "formal_data_cleaning_report.csv"
)

pd.DataFrame({
    "Report": report
}).to_csv(
    report_csv,
    index=False
)


# ============================================================
# STEP 28: FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("DATA CLEANING COMPLETED")
print("=" * 70)

print(
    f"Initial records       : {initial_rows}"
)

print(
    f"Final records         : {final_rows}"
)

print(
    f"Records removed       : {total_removed}"
)

print(
    f"Duplicate rows left   : {final_duplicate_rows}"
)

print(
    f"Duplicate IDs left    : {final_duplicate_ids}"
)

print(
    f"Missing values left   : {final_missing.sum()}"
)

print(
    f"Data quality status   : {status}"
)

print("\nGenerated files:")

print(
    "1.",
    cleaned_file
)

print(
    "2.",
    report_file
)

print(
    "3.",
    report_csv
)

print(
    "4.",
    raw_backup
)

print("=" * 70)

MATERIALS PROJECT DATA CLEANING

Reading file...
File loaded successfully!

Original columns:
0: 'material_id'
1: 'chemical_formula'
2: 'crystal_system'
3: 'space_group'
4: 'full_structure'

Initial shape:
(10000, 5)

Standardized columns:
0: material_id
1: chemical_formula
2: crystal_system
3: space_group
4: full_structure

AUTOMATIC COLUMN DETECTION
Material ID column : material_id
Formula column     : chemical_formula
Structure column   : full_structure

Raw backup saved:
materials_project_cleaning\raw_backup.csv

MISSING VALUES BEFORE CLEANING
Series([], dtype: int64)

Exact duplicate rows: 0
Records with duplicate material IDs: 0

Validating chemical formulas...
Invalid formulas: 0

Missing structures: 0


DATA CLEANING COMPLETED
Initial records       : 10000
Final records         : 10000
Records removed       : 0
Duplicate rows left   : 0
Duplicate IDs left    : 0
Missing values left   : 0
Data quality status   : PASSED

Generated files:
1. materials_project_cleaning\materials_cl